In [0]:
dbutils.widgets.text("quarter", "2024q1")
QUARTER = dbutils.widgets.get("quarter").strip()

CATALOG = "hindsight_dev"
VOLUME  = f"/Volumes/{CATALOG}/bronze/raw"
CONTROL = f"{CATALOG}.ops.ingest_control"
HEADERS = {"User-Agent": "Utkarsh Saraogi utkarsh@example.com"}

WANTED = ["sub.txt", "num.txt", "tag.txt", "pre.txt"]
print("quarter:", QUARTER)

In [0]:
row = spark.sql(
    f"SELECT status, zip_url FROM {CONTROL} WHERE quarter = '{QUARTER}'"
).collect()

if not row:
    raise RuntimeError(f"{QUARTER} not in control table. Run 00_discover_quarters.")

status, zip_url = row[0]["status"], row[0]["zip_url"]

if status == "DONE":
    print(f"{QUARTER} already DONE -- skipping")
    dbutils.notebook.exit("SKIPPED")

spark.sql(f"UPDATE {CONTROL} SET status='RUNNING', attempts=attempts+1 "
          f"WHERE quarter='{QUARTER}'")
print("proceeding:", zip_url)

In [0]:
import io, os, zipfile, requests, time, shutil

dest = f"{VOLUME}/quarter={QUARTER}"

try:
    # download with retry -- SEC occasionally throttles
    for attempt in range(3):
        try:
            resp = requests.get(zip_url, headers=HEADERS, timeout=300)
            resp.raise_for_status()
            break
        except Exception as e:
            if attempt == 2:
                raise
            print(f"  retry {attempt+1}: {e}")
            time.sleep(5 * (attempt + 1))

    nbytes = len(resp.content)
    print(f"downloaded {nbytes/1e6:.1f} MB")

    # write fresh -- partial prior runs must not linger
    if os.path.exists(dest):
        shutil.rmtree(dest)
    os.makedirs(dest, exist_ok=True)

    counts = {}
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        present = set(z.namelist())
        missing = [f for f in WANTED if f not in present]
        if missing:
            raise RuntimeError(f"missing from zip: {missing}")

        for fname in WANTED:
            data = z.read(fname)
            with open(f"{dest}/{fname}", "wb") as fh:
                fh.write(data)
            # line count minus header -- reconciliation baseline for Bronze
            counts[fname] = max(data.count(b"\n") - 1, 0)
            print(f"  {fname}: {counts[fname]:,} rows")

    cnt_sql = ", ".join(f"'{k}', {v}" for k, v in counts.items())
    spark.sql(f"""
        UPDATE {CONTROL}
        SET status='DONE', bytes_downloaded={nbytes},
            files_extracted={len(WANTED)}, row_counts=map({cnt_sql}),
            last_error=NULL, completed_at=current_timestamp()
        WHERE quarter='{QUARTER}'
    """)
    print(f"{QUARTER} DONE")

except Exception as e:
    err = str(e).replace("'", "")[:900]
    spark.sql(f"UPDATE {CONTROL} SET status='FAILED', last_error='{err}' "
              f"WHERE quarter='{QUARTER}'")
    raise